In [2]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow
import unicodedata



# Carregando variáveis de ambiente
load_dotenv()

True

In [3]:
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
project_id = os.getenv("PROJECT_ID")
table_bronze2015 = os.getenv("BRONZE_2015")
table_bronze2016 = os.getenv("BRONZE_2016")
table_bronze2017 = os.getenv("BRONZE_2017")
table_bronze2018 = os.getenv("BRONZE_2018")
table_bronze2019 = os.getenv("BRONZE_2019")
table_bronze2020 = os.getenv("BRONZE_2020")
table_bronze2021 = os.getenv("BRONZE_2021")
table_bronze2022 = os.getenv("BRONZE_2022")
table_bronze2023 = os.getenv("BRONZE_2023")
table_bronze2024 = os.getenv("BRONZE_2024")
table_bronze2025 = os.getenv("BRONZE_2025")


In [4]:
# Inicializa o cliente
client = bigquery.Client.from_service_account_json(credencial, project=project_id)

# Mapeia nome da tabela → nome do DataFrame
tabelas = {
    "df_2015": table_bronze2015,
    "df_2016": table_bronze2016,
    "df_2017": table_bronze2017,
    "df_2018": table_bronze2018,
    "df_2019": table_bronze2019,
    "df_2020": table_bronze2020,
    "df_2021": table_bronze2021,
    "df_2022": table_bronze2022,
    "df_2023": table_bronze2023,
    "df_2024": table_bronze2024,
    "df_2025": table_bronze2025,
}


In [5]:
# Carrega todas as tabelas
dataframes = {}
for nome, table_stg in tabelas.items():
    print(f"Carregando {nome}...")
    query = f"SELECT * FROM `{table_stg}`"
    dataframes[nome] = client.query(query).to_dataframe()

print("✅ Todos os DataFrames carregados!")

Carregando df_2015...


/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Carregando df_2016...
Carregando df_2017...
Carregando df_2018...
Carregando df_2019...
Carregando df_2020...
Carregando df_2021...
Carregando df_2022...
Carregando df_2023...
Carregando df_2024...
Carregando df_2025...
✅ Todos os DataFrames carregados!


In [6]:
df_final = pd.concat(dataframes.values(), ignore_index=True)

In [7]:
df_final.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,26/01/2015,"2,58","2,4337",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,02/02/2015,"2,74","2,4337",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,11/02/2015,"2,88","2,5941",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,18/02/2015,"2,88","2,5941",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
4,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,25/02/2015,"2,87","2,5941",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.


In [8]:
df_final.tail()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
320551,NE,PE,IGARASSU,CEMOPEL CM PETROLEO LTDA,69943686000401,AVENIDA BARAO VERA CRUZ,975,None,CRUZ DE REBOUCAS,53610-296,GNV,02/12/2025,"4,39",NaN,R$ / m3,BRANCA
320552,NE,PE,IGARASSU,CEMOPEL CM PETROLEO LTDA,69943686000401,AVENIDA BARAO VERA CRUZ,975,None,CRUZ DE REBOUCAS,53610-296,GNV,10/12/2025,"4,39",NaN,R$ / m3,BRANCA
320553,NE,PE,IGARASSU,CEMOPEL CM PETROLEO LTDA,69943686000401,AVENIDA BARAO VERA CRUZ,975,None,CRUZ DE REBOUCAS,53610-296,GNV,18/12/2025,"4,39",NaN,R$ / m3,BRANCA
320554,NE,PE,IGARASSU,CEMOPEL CM PETROLEO LTDA,69943686000401,AVENIDA BARAO VERA CRUZ,975,None,CRUZ DE REBOUCAS,53610-296,GNV,22/12/2025,"4,39",NaN,R$ / m3,BRANCA
320555,NE,PE,IGARASSU,CEMOPEL CM PETROLEO LTDA,69943686000401,AVENIDA BARAO VERA CRUZ,975,None,CRUZ DE REBOUCAS,53610-296,GNV,30/12/2025,"4,39",NaN,R$ / m3,BRANCA


In [9]:
df_final.columns = (df_final.columns
                          .str.lower()
                          .str.replace('[ -]', '_', regex=True)
                          .str.replace('_+', '_', regex=True))

In [10]:
df_final.head()

,regiao_sigla,estado_sigla,municipio,revenda,cnpj_da_revenda,nome_da_rua,numero_rua,complemento,bairro,cep,produto,data_da_coleta,valor_de_venda,valor_de_compra,unidade_de_medida,bandeira
0,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,26/01/2015,"2,58","2,4337",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,02/02/2015,"2,74","2,4337",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,11/02/2015,"2,88","2,5941",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,18/02/2015,"2,88","2,5941",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
4,NE,PE,PETROLINA,MANOEL JOSINO NETO & CIA LTDA,00.199.825/0001-87,AVENIDA DAS NACOES,170,None,CENTRO,56304-360,DIESEL,25/02/2015,"2,87","2,5941",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.


In [11]:
df_final = df_final.apply(lambda col: col.map(remove_acentos) if col.dtype == 'object' else col)

NameError: name 'remove_acentos' is not defined

In [ ]:
def remove_acentos(texto):
    if isinstance(texto, str):
        return unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    return texto

In [ ]:
df_final = df_final.apply(lambda col: col.str.lower() if col.dtype == 'object' else col)

In [ ]:
df_final.head()

,regiao_sigla,estado_sigla,municipio,revenda,cnpj_da_revenda,nome_da_rua,numero_rua,complemento,bairro,cep,produto,data_da_coleta,valor_de_venda,valor_de_compra,unidade_de_medida,bandeira
0,ne,pe,petrolina,manoel josino neto & cia ltda,00.199.825/0001-87,avenida das nacoes,170,None,centro,56304-360,diesel,26/01/2015,"2,58","2,4337",r$ / litro,petrobras distribuidora s.a.
1,ne,pe,petrolina,manoel josino neto & cia ltda,00.199.825/0001-87,avenida das nacoes,170,None,centro,56304-360,diesel,02/02/2015,"2,74","2,4337",r$ / litro,petrobras distribuidora s.a.
2,ne,pe,petrolina,manoel josino neto & cia ltda,00.199.825/0001-87,avenida das nacoes,170,None,centro,56304-360,diesel,11/02/2015,"2,88","2,5941",r$ / litro,petrobras distribuidora s.a.
3,ne,pe,petrolina,manoel josino neto & cia ltda,00.199.825/0001-87,avenida das nacoes,170,None,centro,56304-360,diesel,18/02/2015,"2,88","2,5941",r$ / litro,petrobras distribuidora s.a.
4,ne,pe,petrolina,manoel josino neto & cia ltda,00.199.825/0001-87,avenida das nacoes,170,None,centro,56304-360,diesel,25/02/2015,"2,87","2,5941",r$ / litro,petrobras distribuidora s.a.


In [ ]:
df_final['municipio'].value_counts().sort_values()

municipio
afogados da ingazeira         352
sao lourenco da mata          448
sao bento do una              492
bom conselho                  507
sertania                     1354
gravata                      2769
abreu e lima                 2934
camaragibe                   2936
salgueiro                    8209
goiana                       8807
lajedo                       9956
belo jardim                  9967
cabo de santo agostinho      9993
arcoverde                   10709
serra talhada               11710
santa cruz do capibaribe    12704
araripina                   13182
igarassu                    13919
vitoria de santo antao      14264
olinda                      17940
garanhuns                   18554
paulista                    19451
jaboatao dos guararapes     22013
petrolina                   24168
caruaru                     26359
recife                      56859
Name: count, dtype: int64

In [ ]:
df_final['municipio'].drop_duplicates().sort_values().values


array(['abreu e lima', 'afogados da ingazeira', 'araripina', 'arcoverde',
       'belo jardim', 'bom conselho', 'cabo de santo agostinho',
       'camaragibe', 'caruaru', 'garanhuns', 'goiana', 'gravata',
       'igarassu', 'jaboatao dos guararapes', 'lajedo', 'olinda',
       'paulista', 'petrolina', 'recife', 'salgueiro',
       'santa cruz do capibaribe', 'sao bento do una',
       'sao lourenco da mata', 'serra talhada', 'sertania',
       'vitoria de santo antao'], dtype=object)

| Cidade                         | Região                |
|--------------------------------|----------------------|
| Abreu e Lima                   | Metropolitana (RMR)  |
| Cabo de Santo Agostinho        | Metropolitana (RMR)  |
| Camaragibe                     | Metropolitana (RMR)  |
| Igarassu                       | Metropolitana (RMR)  |
| Jaboatão dos Guararapes        | Metropolitana (RMR)  |
| Olinda                         | Metropolitana (RMR)  |
| Paulista                       | Metropolitana (RMR)  |
| Recife                         | Metropolitana (RMR)  |
| São Lourenço da Mata           | Metropolitana (RMR)  |


| Cidade                         | Região                |
|--------------------------------|----------------------|
| Belo Jardim                    | Agreste              |
| Bom Conselho                   | Agreste              |
| Caruaru                        | Agreste              |
| Garanhuns                      | Agreste              |
| Gravatá                        | Agreste              |
| Lajedo                         | Agreste              |
| Santa Cruz do Capibaribe       | Agreste              |
| São Bento do Una               | Agreste              |

| Cidade                         | Região                |
|--------------------------------|----------------------|
| Afogados da Ingazeira          | Sertão               |
| Araripina                      | Sertão               |
| Arcoverde                      | Sertão               |
| Petrolina                      | Sertão               |
| Salgueiro                      | Sertão               |
| Serra Talhada                  | Sertão               |
| Sertânia                       | Sertão               |


| Cidade                         | Região                |
|--------------------------------|----------------------|
| Goiana                         | Zona da Mata Norte   |
| Vitória de Santo Antão         | Zona da Mata Sul     |

In [ ]:
df_final['revenda'].value_counts().sort_values()

revenda
protazio pedro da silva                     2
mf marina club ltda                         2
nvc comercio de combustiveis ltda           2
posto pai & filho ltda - me                 3
auto posto aurora ltda                      3
                                         ... 
albuquerque pneus ltda                   2634
auto posto gene ltda                     3441
cemopel cm petroleo ltda                 3837
petrocal petroleo cavalcanti limitada    4109
caculinha combustiveis ltda              4466
Name: count, Length: 876, dtype: int64

In [ ]:
df_final['bandeira'].drop_duplicates().sort_values().values


array(['ale', 'ale combustiveis', 'alesat', 'alvo', 'branca', 'cbpi',
       'cosan', 'cosan lubrificantes', 'dislub', 'dppi', 'ello',
       'ello-puma', 'fan', 'federal', 'federal energia', 'ipiranga',
       'larco', 'petrobahia', 'petrobras distribuidora s.a.',
       'petrox distribuidora', 'raizen', 'raizen taruma', 'satelite',
       'seta', 'setta distribuidora', 'sp', 'tdc distribuidora', 'temape',
       'torrao', 'vibra', 'vibra energia'], dtype=object)

| Empresa                          | Grupo / Tipo                               |
|----------------------------------|--------------------------------------------|
| Petrobras Distribuidora S.A.     | Grande Distribuidora (BR/Vibra)            |
| Vibra                            | Grande Distribuidora                       |
| Vibra Energia                    | Grande Distribuidora                       |
| Ipiranga                         | Grande Distribuidora                       |
| Raízen                           | Grande Distribuidora                       |
| Raízen Tarumã                    | Grande Distribuidora (Filial Raízen)       |
| Cosan                            | Holding / Controladora (Raízen)            |
| Ale                              | Distribuidora Média                        |
| Ale Combustíveis                 | Distribuidora Média                        |
| Alesat                           | Distribuidora Média                        |
| Petrobahia                       | Distribuidora Regional                     |
| Larco                            | Distribuidora Regional                     |
| Dislub                           | Distribuidora Regional                     |
| DPPi                             | Distribuidora Regional                     |
| Petrox Distribuidora             | Distribuidora Regional                     |
| Setta Distribuidora              | Distribuidora Regional ⚠️                  |
| TDC Distribuidora                | Distribuidora Regional                     |
| Temape                           | Distribuidora / Logística                  |
| Cosan Lubrificantes              | Lubrificantes (Grupo Cosan)                |
| Federal                          | Energia / Combustíveis                     |
| Federal Energia                  | Energia / Combustíveis                     |
| Fan                              | Distribuidora Regional                     |
| Ello                             | Distribuidora Regional ⚠️                  |
| Ello-Puma                        | Distribuidora Regional ⚠️                  |
| Seta                             | Distribuidora Regional ⚠️                  |
| Satélite                         | Distribuidora Regional                     |
| CBPI                             | Distribuidora Regional (Grupo Ipiranga)    |
| Alvo                             | Distribuidora Regional                     |
| Branca                           | Distribuidora Regional                     |
| SP                               | Não identificado / Genérico                |
| Torrão                           | Distribuidora Regional                     |